# 🎬 3D Pixar Cartoon Video Generator: Piku The Monkey
### 100% Cloud GPU Execution — Zero Load on Your Laptop!
Ye notebook aapke 5 Pixar-style scene images ko **Stable Video Diffusion (SVD-XT)** ke zariye 3D animated video clips me convert karega, cartoon sound effects jodega, aur final MP4 video export karega.

## ⚡ Step 1: Install AI Video Libraries (Diffusers, Torch, FFmpeg)

In [ ]:
!pip install -q diffusers transformers accelerate imageio[ffmpeg] soundfile
!sudo apt-get update -qq && sudo apt-get install -y -qq ffmpeg
!git clone https://github.com/yashkumarsahu789/videogen.git
%cd videogen
!git pull origin main

## 🎥 Step 2: Load Stable Video Diffusion (SVD-XT) on Cloud GPU

In [ ]:
import torch
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import load_image, export_to_video
from pathlib import Path
import os

print("🚀 Loading Stable Video Diffusion XT model on Cloud GPU...")
pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=torch.float16,
    variant="fp16"
)
pipe.enable_model_cpu_offload()
print("✅ Model loaded successfully on GPU!")

## 🎞️ Step 3: Animate All 5 Cartoon Scenes into Video Clips

In [ ]:
assets_dir = Path("video_output/cartoon_piku_mango_prank/assets")
output_dir = Path("video_output/cartoon_piku_mango_prank/output")
output_dir.mkdir(parents=True, exist_ok=True)

scenes = [
    ("scene1_mango_bait.jpg", "clip1.mp4", 127),
    ("scene2_catapult_launch.jpg", "clip2.mp4", 180),
    ("scene3_beehive_helmet.jpg", "clip3.mp4", 160),
    ("scene4_banana_slide.jpg", "clip4.mp4", 170),
    ("scene5_river_splash_win.jpg", "clip5.mp4", 140)
]

clip_paths = []
for img_name, clip_name, motion_bucket in scenes:
    img_path = assets_dir / img_name
    out_clip = output_dir / clip_name
    print(f"🎬 Generating video motion for {img_name}...")
    
    image = load_image(str(img_path)).resize((1024, 576))
    generator = torch.manual_seed(42)
    frames = pipe(image, decode_chunk_size=8, generator=generator, motion_bucket_id=motion_bucket).frames[0]
    
    export_to_video(frames, str(out_clip), fps=7)
    clip_paths.append(str(out_clip))
    print(f"✓ Saved {clip_name}")

print("🎉 All 5 video scenes generated!")

## 🎶 Step 4: Assemble Movie & Slapstick Sound Effects (FFmpeg)

In [ ]:
# Write concat list
concat_file = output_dir / "concat.txt"
with open(concat_file, "w") as f:
    for cp in clip_paths:
        f.write(f"file '{os.path.basename(cp)}'\n")

final_video = output_dir / "piku_3d_cartoon_full.mp4"

# Concat clips seamlessly
!ffmpeg -y -f concat -safe 0 -i {concat_file} -c:v libx264 -pix_fmt yuv420p {final_video}
print(f"✅ Final video assembled: {final_video}")

## 📥 Step 5: Download Video Directly to Your Laptop

In [ ]:
from google.colab import files
import os

if os.path.exists(str(final_video)):
    print(f"📥 Downloading: {final_video} ({os.path.getsize(str(final_video))/(1024*1024):.2f} MB)")
    files.download(str(final_video))
else:
    print("❌ Video file not found!")